# EXAONE-3.5-7.8B-Instruct 스왑 스모크 테스트

`adapters.todo_creation.qwen_llm.QwenLLM` 은 OpenAI 호환 `/chat/completions` 만 호출하므로 **모델 비종속**이다.
→ Qwen 을 EXAONE 으로 바꾸는 데 어댑터 코드 변경이 필요 없다. `model` 과 엔드포인트만 EXAONE 으로 바꾸면 된다.

이 노트북이 검증하는 것 (실제 파이프라인 메서드 그대로 호출):
- `split_tasks` — 뉴로-심볼릭 분해기가 파싱 가능한 JSON 을 내는가
- `judge_sufficiency` → `generate_plan` — 충분성 판단 후 플랜 JSON 이 파싱되는가

## 두 가지 실행 경로
1. **엔드포인트 경로 (권장, 신규 코드 0줄)** — vLLM 으로 EXAONE 을 띄우고 `QwenLLM` 이 그대로 붙는다. 운영(RunPod vLLM) 과 동일 경로.
   ```bash
   vllm serve LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct \
       --trust-remote-code --served-model-name LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct
   ```
2. **로컬 transformers 경로 (서버 없이)** — 노트북 안에서 모델을 직접 로드. guided_json(한국어-only 제약) 은 못 쓰므로 순수 스모크용.

아래 `MODE` 로 둘 중 하나를 고른다.

In [ ]:
import os
import sys
from datetime import date
from pathlib import Path
from pprint import pprint

from dotenv import load_dotenv

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

from adapters.todo_creation.qwen_llm import QwenLLM

EXAONE_MODEL = "LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct"
TODAY = date.today()

# "endpoint" = vLLM OpenAI 호환 서버에 붙음 / "local" = transformers 로 인프로세스 로드
MODE = os.getenv("EXAONE_TEST_MODE", "endpoint")
print("ROOT:", ROOT, "| TODAY:", TODAY, "| MODE:", MODE)

## LLM 인스턴스 만들기
- `endpoint`: 신규 코드 없이 `QwenLLM(base_url=..., model=EXAONE)`.
- `local`: `complete_raw` 만 transformers 호출로 오버라이드(서명/파싱은 그대로 상속).

In [ ]:
if MODE == "endpoint":
    # base_url 예: http://localhost:8000/v1
    base_url = os.getenv("EXAONE_BASE_URL", "http://localhost:8000/v1")
    llm = QwenLLM(base_url=base_url, model=EXAONE_MODEL, api_key=os.getenv("EXAONE_API_KEY", "EMPTY"))
    print("endpoint:", base_url, "| model:", EXAONE_MODEL)

elif MODE == "local":
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    # EXAONE 은 커스텀 아키텍처라 trust_remote_code=True 필요 (transformers>=4.43 권장).
    _tok = AutoTokenizer.from_pretrained(EXAONE_MODEL, trust_remote_code=True)
    _model = AutoModelForCausalLM.from_pretrained(
        EXAONE_MODEL, trust_remote_code=True, torch_dtype="auto", device_map="auto"
    )

    class ExaoneLocalLLM(QwenLLM):
        """complete_raw 만 로컬 generate 로 대체. guided_json 은 무시(스모크용)."""

        async def complete_raw(self, *, messages, label="exaone", temperature=None, guided_json=None):
            temp = self.temperature if temperature is None else temperature
            input_ids = _tok.apply_chat_template(
                messages, add_generation_prompt=True, return_tensors="pt"
            ).to(_model.device)
            with torch.no_grad():
                out = _model.generate(
                    input_ids,
                    max_new_tokens=self.max_tokens,
                    do_sample=temp > 0,
                    temperature=max(temp, 1e-4),
                    top_p=self.top_p,
                    eos_token_id=_tok.eos_token_id,
                    pad_token_id=_tok.pad_token_id or _tok.eos_token_id,
                )
            return _tok.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True)

    llm = ExaoneLocalLLM(base_url="", model=EXAONE_MODEL)
    print("local model loaded:", EXAONE_MODEL)

else:
    raise ValueError(f"MODE 는 'endpoint' 또는 'local': got {MODE!r}")

## 1. 연결/JSON 스모크
EXAONE 이 JSON 한 덩어리를 뱉고 envelope 가 정상인지 최소 확인.

In [ ]:
raw = await llm.complete_raw(
    messages=[
        {"role": "system", "content": "너는 JSON 만 출력하는 어시스턴트다."},
        {"role": "user", "content": '{"ping": true} 를 그대로 한 번 출력하라.'},
    ],
    label="smoke",
)
print(repr(raw[:300]))

## 2. split_tasks — 뉴로-심볼릭 분해기
실제 `TASK_SPLITTER_SYSTEM` 프롬프트 + 파서/재시도 그대로 사용.

In [ ]:
res = await llm.split_tasks(prompt="내일까지 발표자료 만들고 다음주 월요일에 리허설 하기", today=TODAY)
print("intent:", res.intent)
for t in res.tasks:
    print(f"  - {t.title} | due={t.due_date} | tags={t.tags}")

## 3. judge_sufficiency → generate_plan
충분성 판단 후 플랜 JSON 까지 파이프라인대로 흘려본다.

> `endpoint` 모드에서는 `generate_plan` 이 `guided_json`(한국어-only 제목) 을 보낸다. EXAONE 을 띄운 vLLM 이 outlines 백엔드를 지원해야 통과한다. 미지원이면 이 셀에서 에러가 나는데, 그건 *제목 제약* 문제지 모델 호환성 문제는 아니다.

In [ ]:
history = []
message = "3개월 안에 정보처리기사 필기 합격하고 싶어. 평일 저녁 1시간 정도 공부 가능해."
is_sufficient, missing, goal = await llm.judge_sufficiency(history=history, message=message, today=TODAY)
print("sufficient:", is_sufficient, "| missing:", missing)
pprint(dict(goal))

if is_sufficient:
    summary, days = await llm.generate_plan(parsed_goal=goal, today=TODAY)
    print("\nsummary:", summary)
    for d in days[:5]:
        titles = ", ".join(t.title for t in d["tasks"])
        print(f"  {d['date']}: {titles}")
    print(f"... 총 {len(days)} 일")
else:
    print("정보 부족 — 꼬리질문 경로")

## 체크리스트
- [ ] 1~3 셀이 파싱 에러 없이 통과 → EXAONE 이 현행 프롬프트/파서와 호환
- [ ] `generate_plan` 제목이 한국어 위주인가 (guided_json 없으면 영어 섞일 수 있음 → 운영 vLLM 에 outlines 백엔드 확인)
- [ ] EXAONE-3.5 는 비상업 연구 라이선스 — 배포 전 라이선스 확인

통과하면 RunPod 워커(`runpod_workers/llm`) 의 베이스 모델/`bake.py` 를 EXAONE 으로 교체하는 게 다음 단계.